# Workspace Bootstrap (Colab)

Notebook template nay chi chua code mau. Du lieu thuc duoc lay tu backend qua launch token ngan han.

In [ ]:
import os
import requests

API_BASE = os.getenv("API_BASE", "")
if not API_BASE:
    try:
        from google.colab import output
        API_BASE = output.eval_js("new URL(window.location.href).searchParams.get('api_base_url')") or ""
    except Exception:
        pass
if not API_BASE:
    API_BASE = input("Paste NeuralSpace public API base URL: ").strip()
API_BASE = API_BASE.rstrip("/")
print("API_BASE =", API_BASE)

In [ ]:
launch_token = ""

try:
    from google.colab import output
    token_from_url = output.eval_js("new URL(window.location.href).searchParams.get('launch_token')")
    if token_from_url:
        launch_token = token_from_url
except Exception:
    pass

if not launch_token:
    launch_token = input("Paste launch_token from your app: ").strip()

if not launch_token:
    raise RuntimeError("Missing launch_token")

print("launch_token received")

In [ ]:
bootstrap = requests.post(
    f"{API_BASE}/colab/bootstrap",
    json={"token": launch_token},
    timeout=30,
)
bootstrap.raise_for_status()
cfg = bootstrap.json()
RUNTIME_TOKEN = cfg["runtime_token"]
RUNTIME_HEADERS = {"Authorization": f"Bearer {RUNTIME_TOKEN}"}
print("session_id:", cfg["session_id"])
print("workspace_id:", cfg["workspace_id"])
print("capabilities:", cfg["capabilities"])
print("datasets:", len(cfg.get("datasets", [])))

In [ ]:
def heartbeat():
    response = requests.post(f"{API_BASE}/colab/runtime/heartbeat", headers=RUNTIME_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

def log_metrics(run_id, **values):
    response = requests.post(
        f"{API_BASE}/colab/runtime/runs/{run_id}/metrics",
        headers=RUNTIME_HEADERS, json={"values": values}, timeout=30,
    )
    response.raise_for_status()
    return response.json()

heartbeat()

In [ ]:
import pandas as pd

if not cfg.get("datasets"):
    print("No datasets attached to this workspace")
else:
    first_ds = cfg["datasets"][0]
    print("Loading:", first_ds["dataset_id"], first_ds["name"])
    df = pd.read_csv(first_ds["signed_url"])
    print(df.head())